In [1]:
import pandas as pd
from readback_guard.config import load_config
from readback_guard.evaluate import load_cases, gold_clearance, rates, log_result
from readback_guard.extract import extract

cfg = load_config()
dev = load_cases(cfg, "dev")
dev.groupby(["kind", "verdict"]).size()

kind      verdict       
injected  WRONG_CALLSIGN    12
          WRONG_VALUE       23
real      CORRECT           37
          MISSING_ITEM       8
          WRONG_CALLSIGN     3
          WRONG_VALUE        4
dtype: int64

In [2]:
row = dev.iloc[0]
print(row.atco_text)
extract(row.atco_text, cfg)

hotel zulu mike sierra four runway three one cleared for takeoff wind one eight zero degrees seven knots good bye


Clearance(callsign='hzms4', items=[Item(type='runway', value='31', unit=None, qualifier=None)])

In [3]:
texts = pd.unique(pd.concat([dev.atco_text, dev.pilot_text]))
ext = {}
for n, text in enumerate(texts, 1):
    ext[text] = extract(text, cfg)
    if n % 20 == 0:
        print(n, "/", len(texts))
len(ext)

20 / 139
40 / 139
60 / 139
80 / 139
100 / 139
120 / 139


139

In [4]:
from readback_guard.compare import compare

dev["pred"] = [compare(gold_clearance(r.atco_callsign, r.atco_items),
                       gold_clearance(r.pilot_callsign, r.pilot_items)).verdict
               for r in dev.itertuples()]
m_oracle = rates(dev)
m_oracle

{'n_errors': 50,
 'detected': 50,
 'detection_rate': 1.0,
 'detection_ci': (0.929, 1.0),
 'n_correct': 37,
 'false_alarms': 0,
 'false_alarm_rate': 0.0,
 'false_alarm_ci': (0.0, 0.094),
 'exact_verdict_rate': 0.989}

In [5]:
verdicts = [compare(ext[r.atco_text], ext[r.pilot_text]) for r in dev.itertuples()]
dev["pred"] = [v.verdict for v in verdicts]
dev["reason"] = [v.reason for v in verdicts]
m_text = rates(dev)
m_text

ValueError: invalid literal for int() with base 10: 'LEFT'